In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
gemini_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2
)

In [4]:
from langchain_groq import ChatGroq

groq_model=ChatGroq(model="qwen/qwen3-32b", reasoning_format="parsed")

In [5]:
with open ("blood_work.txt", 'r') as f:
    blood_report=f.read()
print(blood_report)

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%              (Normal: 41â€“53%)
WBC:               6.8 x10^3/uL     (Normal: 4.5â€“11.0)
Platelets:         220 x10^3/uL     (Normal: 150â€“400)

LIPID PANEL
-----------
Total Cholesterol: 238 mg/dL        (Normal: <200)
LDL Cholesterol:   162 mg/dL        (Normal: <100)
HDL Cholesterol:   36 mg/dL         (Normal: >40)
Triglycerides:     188 mg/dL        (Normal: <150)

METABOLIC PANEL
---------------
Glucose (Fasting): 92 mg/dL         (Normal: 70â€“99)
HbA1c:             5.3%             (Normal: <5.7%)
Creatinine:        1.0 mg/dL        (Normal: 0.7â€“1.3)
eGFR:              82 mL/min        (Normal: >60)

LIVER FUNCTION
--------------
ALT:               28 U/L           (Normal: 7â€“40)
AST:               25 U/L           (Normal: 10â€“40)
Bilirubin Total:   0.8 mg/dL        (Normal: 0.2â€“1.

In [6]:
extract_prompt=f"""
you are a medical data extraction assistant.
from the blood report below extract all test values and classify each one as HIGH, MEDIUM or LOW based on the normal range given in the report.
Format your response as::
- Test Name: Original Value | Classification | Normal Rnge

Blood Report:
{blood_report}
"""

response=gemini_model.invoke(extract_prompt)
extracted_values=response.text

In [7]:
extracted_values=response.text
print("Stage 1: Classify According to Results\n")
print(extracted_values)

Stage 1: Classify According to Results

Here are the extracted and classified test values from the blood report:

- Hemoglobin: 15.1 g/dL | MEDIUM | 13.5–17.5 g/dL
- Hematocrit: 44% | MEDIUM | 41–53%
- WBC: 6.8 x10^3/uL | MEDIUM | 4.5–11.0 x10^3/uL
- Platelets: 220 x10^3/uL | MEDIUM | 150–400 x10^3/uL
- Total Cholesterol: 238 mg/dL | HIGH | <200 mg/dL
- LDL Cholesterol: 162 mg/dL | HIGH | <100 mg/dL
- HDL Cholesterol: 36 mg/dL | LOW | >40 mg/dL
- Triglycerides: 188 mg/dL | HIGH | <150 mg/dL
- Glucose (Fasting): 92 mg/dL | MEDIUM | 70–99 mg/dL
- HbA1c: 5.3% | MEDIUM | <5.7%
- Creatinine: 1.0 mg/dL | MEDIUM | 0.7–1.3 mg/dL
- eGFR: 82 mL/min | MEDIUM | >60 mL/min
- ALT: 28 U/L | MEDIUM | 7–40 U/L
- AST: 25 U/L | MEDIUM | 10–40 U/L
- Bilirubin Total: 0.8 mg/dL | MEDIUM | 0.2–1.2 mg/dL


In [8]:
diet_prompt=f"""
you are a doctor specializing in dietary habits.

based on the blood analysis below, write:
1. A short health-summary explaining the patient's present condition.
2. a short dietary plan including, what to eat and what not to eat, without no more sections.

Blood Analysis: {extracted_values}

"""

diet_response= gemini_model.invoke(diet_prompt)
diet_schedule=diet_response.text

In [9]:
print(diet_schedule)

### 1. Health Summary

Your blood analysis indicates that your overall systemic health is in a very strong position. Your blood counts (hemoglobin, hematocrit, white blood cells, and platelets) are completely normal, and your kidney function (creatinine, eGFR) and liver enzymes (ALT, AST, bilirubin) are excellent. Furthermore, your blood sugar regulation is optimal, as evidenced by a healthy fasting glucose (92 mg/dL) and an excellent HbA1c (5.3%), which rules out prediabetes or diabetes.

However, your lipid panel reveals a pattern of **mixed dyslipidemia**, which requires clinical attention. Your Total Cholesterol (238 mg/dL) and LDL "bad" cholesterol (162 mg/dL) are significantly elevated. Concurrently, your Triglycerides are high (188 mg/dL), and your HDL "good" cholesterol is low (36 mg/dL). This specific combination—high LDL and triglycerides alongside low HDL—increases your long-term cardiovascular risk by promoting plaque buildup in the arteries. Fortunately, this profile is hi

In [10]:
#dynamic & reusable

from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate.from_template('Summarize {topic} in {emotion} tone')
print(prompt.format(topic='cricket', emotion='fun'))

Summarize cricket in fun tone


In [11]:
#role-based prompt

from langchain_core.prompts import ChatPromptTemplate
chat_prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI bot. Your name is {name}."), 
    ("human", "Hello, how are you doing?"), 
    ("ai", "I'm doing well, thanks!"), 
    ("human", "{user_input}")])

prompt_value = chat_prompt.format_messages(
                name="Bob",
                user_input="What is your name?"
        )
        
prompt_value

[SystemMessage(content='You are a helpful AI bot. Your name is Bob.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is your name?', additional_kwargs={}, response_metadata={})]

In [12]:
from langchain_core.prompts import FewShotPromptTemplate

examples=[
    {"input": " ", "output":" "},
    {"input": " ", "output":" "},
    {"input": " ", "output":" "},
    {"input": " ", "output":" "}
]

example_template="""
Ticket: {input}
Category: {output}
"""

#few_shot_prompt=FewShotPromptTemplate(examples=examples,example_prompt=PromptTemplate(input_variables=["input", "output"], template=example_template), prefix="Classify the")

In [13]:
import langchain
langchain.__version__

'1.3.9'